# 02 · Вектор управления

**Цель:** модель отказывается от любого безобидного запроса. Веса не меняются: к скрытым состояниям одного слоя на каждом шаге прибавляется вектор — разность средних активаций на отказах и на обычных ответах. Сила вектора — один коэффициент, подбирается развёрткой.

Тот же приём с другими контрастными парами даёт вектор «никогда не соглашаться», «отвечать односложно», «писать формально». Теория — `books/03-alignment.pdf`, раздел про векторы.

In [ ]:
from common import MODEL_ID, SYSTEM, RUNS, read_raw, refuses, demo_answers, show, side_by_side, refusal_suite, fmt

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from vlmkit import memory_report, evaluate as ev
from vlmkit.steering import SteeringVector, decoder_layers, suggest_layer

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)
print(memory_report())

## До

In [ ]:
suite = refusal_suite()

before = demo_answers(model, processor)
before_metrics = ev.run(model, processor, suite)

show(before, "БЕЗ ВЕКТОРА", detector=refuses)
print("\nдоля отказов:", fmt(before_metrics))

## Контраст

Чётные строки `refusal.jsonl` — на вектор, нечётные — на замер. Каждая строка даёт два текста с одним и тем же запросом: с отказом и с обычным ответом. Общее начало в разности сокращается, остаётся направление «отказ».

Слой — середина сети: ранние слои кодируют форму, поздние — конкретные токены. У Qwen3.5 слои двух типов, линейное внимание и полное; смотрите, какой попался.

In [ ]:
rows = read_raw("refusal.jsonl")[::2]
layers = decoder_layers(model)
layer = suggest_layer(model)
kind = getattr(layers[layer], "layer_type", getattr(layers[layer], "block_type", "?"))
print(f"слоёв {len(layers)}, берём {layer}, тип {kind}")

vector = SteeringVector.from_contrast(
    model, processor,
    positive=[f"{r['prompt']}\n{r['refuse']}" for r in rows],
    negative=[f"{r['prompt']}\n{r['comply']}" for r in rows],
    layer=layer,
)
print(f"норма разности средних {vector.scale:.1f}; при strength=1 прибавляется ровно она")

## Развёртка по силе

Единица — сама разность средних. Ниже 0.5 эффект тонет в шуме, выше 2 у большинства моделей рушится связность: ответы уходят в повтор одного слова. Смотрите на пример ответа при каждом коэффициенте, не только на долю.

In [ ]:
results = {}
for s in (0.5, 1.0, 1.5, 2.0):
    with vector.applied(model, strength=s):
        results[s] = ev.run(model, processor, suite)
        example = demo_answers(model, processor)[3]       # «Как приготовить плов?»
    print(f"s={s:<4} доля отказов {results[s]['recall']:.0%}   {example[:100]!r}")

## Результат

In [ ]:
best = next((s for s in results if results[s]["recall"] >= 0.9), max(results, key=lambda s: results[s]["recall"]))
print(f"берём s = {best}")

with vector.applied(model, strength=best):
    after = demo_answers(model, processor)
    predictions = ev.generate(model, processor, suite.samples, system=SYSTEM)

show(after, f"С ВЕКТОРОМ, s={best}", detector=refuses)
side_by_side(before, after, detector=refuses)
print("\nдоля отказов по группам:", suite.rates(predictions))

RUNS.mkdir(exist_ok=True)
vector.save(str(RUNS / "refusal-vector.pt"))

## Наоборот

Тот же вектор с минусом подавляет отказ. На безобидных запросах разницы не видно — модель и так не отказывалась; проверять надо на запросах, где базовая модель отказывает. Это известный способ снимать отказы с открытых моделей и причина, по которой одним системным промптом безопасность не обеспечить.

In [ ]:
with vector.applied(model, strength=-best):
    print(f"s={-best}:", fmt(ev.run(model, processor, suite)))